In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, GPT2Config
from transformers import get_linear_schedule_with_warmup

import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split, RandomSampler, SequentialSampler

import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
# model_name: ['gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl']
model_name = "gpt2-medium" 
model_save_path = './model'

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

uintx feature requires torch 2.3+, please upgrade pytorch


In [4]:
configuration = GPT2Config.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, config=configuration)

tokenizer = GPT2TokenizerFast.from_pretrained(model_name)

input_sequence = "beef, salt, pepper"
input_ids = tokenizer.encode(input_sequence, return_tensors='pt')

model = model.to(device)
#combine both sampling techniques
sample_outputs = model.generate(input_ids.to(device),
                              do_sample = True, max_length = 120,
                              top_k = 50, top_p = 0.85,
                              num_return_sequences = 3)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
/Users/giorgos/anaconda3/envs/pytorch_env/lib/python3.11/site-packages/transformers/pytorch_utils.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


In [7]:
tokenizer.decode(sample_outputs[2], skip_special_tokens=True)

'beef, salt, pepper, and ketchup, the beef can be fried, grilled, roasted, or fried up in oil. The beef is cooked in the oil for about 5 minutes before the meat is cut up, served with onions, tomatoes, and other vegetables, or served as a side dish. The steak is grilled to brown the meat, then the meat is grilled again for about 2-3 minutes, before being served with some of the sauce. The beef is then cut into thin pieces and the meat is served with some chopped vegetables. The Beef and Lamb is served with onion,'

In [18]:
from nlp_dataset import generate_sample

def form_string(sample: tuple[list, float]) -> str:
    sample_text = sample[0]
    sample_ans = sample[1]
    prompt = "<|startoftext|>" + ", ".join(str(x) for x in sample_text[:-1]) + ". " + sample_text[-1] + "." + " Answer: " + str(sample_ans)
    return prompt

In [20]:
num_cats = 8
query_type = "min"
num_query_cats = 4
train_low = 0
train_high = 5
test_low = 0
test_high = 20
num_train_samples = 1000
num_test_samples = 100

In [33]:
train_data = [form_string(generate_sample(num_cats, query_type, train_low, train_high, num_query_cats)) for _ in range(num_train_samples)]
test_data = [form_string(generate_sample(num_cats, query_type, test_low, test_high, num_query_cats, train=False)) for _ in range(num_test_samples)]

In [34]:
tokenizer = GPT2TokenizerFast.from_pretrained(model_name,
                                              bos_token='<|startoftext|>',
                                              eos_token='<|endoftext|>',
                                              unk_token='<|unknown|>',
                                              pad_token='<|pad|>'
                                             )

In [38]:
batch_size = 2
max_length = 180  

# standard PyTorch approach of loading data in using a Dataset class.
class NAR_Dataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.input_ids = []
        self.attn_masks = []

        for recipe in data:
            encodings = tokenizer.encode_plus(recipe,
                                              truncation=True,
                                              padding='max_length',
                                              max_length=max_length,
                                              # return a PyTorch tensor
                                              return_tensors='pt'       
                                             )
            self.input_ids.append(torch.squeeze(encodings['input_ids'],0))
            self.attn_masks.append(torch.squeeze(encodings['attention_mask'],0))


    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        return self.input_ids[idx], self.attn_masks[idx]

dataset_indist = NAR_Dataset(train_data, tokenizer)
dataset_ood = NAR_Dataset(test_data, tokenizer)
print(f"input_ids: {dataset_ood[0][0]} attn_masks: {dataset_ood[0][1]}")

input_ids: tensor([50257, 21979,   132,   232,    11,  1478,    13,    16,    11,  5181,
          132,   235,    11,  1315,    13,  1954,    11,  5181,   132,   238,
           11,  1367,    13,  4089,    11,  5181,   132,   241,    11,  5181,
            9,   132,   238,    11,  1315,    13,  2548,    11,  5181,   132,
          242,    11,  1315,    13,  1731,    11,  5181,   132,   243,    11,
         1478,    13,  4310,    11,  5181,   132,   248,    11,  5181,     9,
          132,   250,    11,  1315,    13,  3324,    11,  5181,   132,   250,
           11,  1467,    13,  4310,    13,  9938,   949,   286,  9376,  5181,
          132,   248,    11,  5181,   132,   243,    11,  5181,   132,   250,
          290,  5181,   132,   235,    13, 23998,    25,  1478,    13,  4310,
        50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259,
        50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259,
        50259, 50259, 50259, 50259, 50259, 50259, 502

In [39]:
# Split into training and validation sets
train_size = int(0.9 * len(dataset_indist))
val_size = len(dataset_indist) - train_size

train_dataset, val_dataset = random_split(dataset_indist, [train_size, val_size])

# Create the DataLoaders for our training and validation datasets.
# Get training samples in random order.
train_dataloader = DataLoader(
            train_dataset, 
            sampler = RandomSampler(train_dataset),
            batch_size = batch_size # Trains with this batch size.
        )

# Get valiation samples sequentially.
validation_dataloader = DataLoader(
            val_dataset, 
            sampler = SequentialSampler(val_dataset),
            batch_size = batch_size # Evaluate with this batch size.
        )

test_dataloader = DataLoader(
            dataset_ood, 
            sampler = SequentialSampler(dataset_ood),
            batch_size = batch_size # Evaluate with this batch size.
        )
            


In [40]:
configuration = GPT2Config.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, config=configuration)
model = model.to(device)
model.resize_token_embeddings(len(tokenizer))

epochs = 3
learning_rate = 2e-5
warmup_steps = 1e2
# to prevent any division by zero in the implementation
epsilon = 1e-8
optim = AdamW(model.parameters(), lr = learning_rate, eps = epsilon)

total_steps = len(train_dataloader) * epochs  # [no batches] x [no epochs]

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optim,
                                            num_warmup_steps=warmup_steps,
                                            num_training_steps=total_steps)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [41]:
def infer(prompt):
    input = f"<|startoftext|>Ingredients: {prompt.strip()}"
    input = tokenizer(input, return_tensors="pt")
    input_ids      = input["input_ids"]
    attention_mask = input["attention_mask"]

    output = model.generate(input_ids.to(device),
                            attention_mask=attention_mask.to(device),
                            max_new_tokens=max_length,
                            do_sample = True, top_k = 50, top_p = 0.85)
    output = tokenizer.decode(output[0], skip_special_tokens=True)
    return output

In [44]:
for epoch_i in range(0, epochs):
    total_train_loss = 0
    model.train() 

    for step, batch in enumerate(train_dataloader): 
        b_input_ids = batch[0].to(device) 
        b_labels    = batch[0].to(device)
        b_masks     = batch[1].to(device) 

        model.zero_grad()
        outputs = model( input_ids = b_input_ids, labels = b_labels,
                         attention_mask = b_masks, token_type_ids = None )

        loss = outputs[0]

        # Get sample every x batches.
        if step % 1 == 0 and not step == 0:
            model.eval()
            print(infer(test_data[0]))
            model.train()

        loss.backward()
        optim.step()
        scheduler.step()

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
/Users/giorgos/anaconda3/envs/pytorch_env/lib/python3.11/site-packages/transformers/pytorch_utils.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


KeyboardInterrupt: 